# E4: Pipeline Control and Component Ablation

**Work Plan Reference:** Deliverable III · F26-066 (Mixture of Experts Framework for Domain Adaptive Colorization)
**Owner:** Person B (Extraction, Colorization Wrapper, Blending, Config File, and E4 Experiments)

---

### Experiment E4 Overview
Evaluate the full pipeline with identical experts compared with the E3 baseline, changing one component at a time:
1. **Identical Experts:** Four domain slots (`people`, `nature`, `urban`, `animals`) plus `fallback`, all loading the same pretrained DDColor-T model (ConvNeXt-T backbone, 256×256 input, ab chrominance output).
2. **Context Margin Ablation:** Expanding bounding boxes by context margin $\rho \in \{0.0, 0.1, 0.2, 0.3\}$.
3. **Blending Switch:** Soft Gaussian-weighted blending (`blending_on=True`) versus hard binary mask paste (`blending_on=False`).
4. **Smoothing Width Ablation:** Gaussian standard deviation $\sigma_0 \in \{2.0, 5.0, 10.0, 15.0, 20.0\}$ px.
5. **Region Shape:** SAM masks (`region_shape='mask'`) versus bounding boxes (`region_shape='box'`).
6. **E3 Baseline Comparison:** DDColor-T on whole image without decomposition.

**Metrics Evaluated:** PSNR, SSIM, Lab $\Delta E$, Boundary $\Delta E$, Colorfulness, LPIPS.

---

### Kaggle Settings (Right-Hand Panel):
- **Accelerator:** GPU T4 x2 (or P100)
- **Internet:** **ON** (required for pip install & HuggingFace weights download)
- **Dataset:** In Kaggle click **Add Data -> Datasets** and search for [`dntai2/cocostuf-2017`](https://www.kaggle.com/datasets/dntai2/cocostuf-2017) (mounted at `/kaggle/input/cocostuf-2017`).

## 1. Clone or Update Repository

In [ ]:
import os
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/zaeem886/domain-adaptive-colorization.git"
BRANCH = "B"  # Person B branch
REPO_DIR = Path("/kaggle/working/domain-adaptive-colorization")

if (REPO_DIR / ".git").is_dir():
    print("Updating existing repository...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=False)
else:
    print(f"Cloning {REPO_URL} (branch {BRANCH})...")
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())

## 2. Install Dependencies

Kaggle environments provide PyTorch and Transformers. Here we install `lpips`, `segment-anything`, `basicsr`, `scikit-image`, and `huggingface-hub`.

In [ ]:
!pip install -q --no-deps lpips
!pip install -q segment-anything transformers timm einops scikit-image scipy opencv-python-headless huggingface-hub matplotlib pandas
print("Dependencies installed successfully.")

## 3. Dataset Setup (Symlink COCO val2017)

The dataset [`dntai2/cocostuf-2017`](https://www.kaggle.com/datasets/dntai2/cocostuf-2017) is mounted at `/kaggle/input/cocostuf-2017`.
We create fast symlinks so that no data is copied into working storage.

In [ ]:
import shutil
from pathlib import Path

SRC = Path("/kaggle/input/cocostuf-2017")
DATASET_ROOT = Path("/kaggle/working/cocostuff_data")
IMAGES_ROOT = DATASET_ROOT / "images"

if SRC.is_dir():
    val_target = SRC / "val2017" / "val2017"
    if not val_target.is_dir():
        val_target = SRC / "val2017"
    link = IMAGES_ROOT / "val2017"
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink() or link.is_file():
        link.unlink()
    elif link.is_dir():
        shutil.rmtree(link)
    link.symlink_to(val_target, target_is_directory=True)
    print("Linked val2017 images to:", link)
else:
    print("Notice: /kaggle/input/cocostuf-2017 not mounted. If running locally or on custom dataset, set IMAGES_ROOT directly.")
    IMAGES_ROOT = Path("pics")

## 4. Download DDColor and Pretrained DDColor-T Model

We clone the official DDColor codebase and fetch the official `piddnad/ddcolor_paper_tiny` pretrained weights (`pytorch_model.bin`) from Hugging Face Hub.

In [ ]:
from huggingface_hub import hf_hub_download
import urllib.request

DDCOLOR_DIR = Path("/kaggle/working/DDColor")
if not DDCOLOR_DIR.exists():
    !git clone --depth 1 https://github.com/piddnad/DDColor.git {DDCOLOR_DIR}

print("Downloading DDColor-T checkpoint from Hugging Face...")
CHECKPOINT = hf_hub_download("piddnad/ddcolor_paper_tiny", "pytorch_model.bin")
print("DDColor repository:", DDCOLOR_DIR)
print("DDColor-T checkpoint:", CHECKPOINT)

# Optional: download SAM ViT-B checkpoint if online Grounded-SAM decomposition is needed
SAM_CHECKPOINT = Path("/kaggle/working/sam_vit_b.pth")
if not SAM_CHECKPOINT.is_file():
    print("Downloading SAM ViT-B weights (~375MB)... ")
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
        str(SAM_CHECKPOINT),
    )
    print("SAM weights downloaded:", SAM_CHECKPOINT)
else:
    print("SAM checkpoint exists:", SAM_CHECKPOINT)

## 5. Common Configuration and Command Runner

Shared CLI runner wrapping `experiments/e4.py`.

In [ ]:
import shlex
import sys

COMMON = [
    "--checkpoint", str(CHECKPOINT),
    "--ddcolor-path", str(DDCOLOR_DIR),
    "--images-root", str(IMAGES_ROOT),
    "--val-split", "val2017",
    "--results-dir", "results/e4",
    "--cache-dir", "cache",
    "--data-lists-dir", "data_lists",
    "--sam-checkpoint", str(SAM_CHECKPOINT),
    "--device", "cuda",
]

def run_e4(*stage_args):
    """Run one experiments/e4.py stage with shared options."""
    command = [sys.executable, "experiments/e4.py", *stage_args, *COMMON]
    print("$", " ".join(shlex.quote(str(part)) for part in command))
    subprocess.run(command, check=True)

## 6. Stage 1: Create Fixed Test ID Lists

Exports the fixed 100-image evaluation set (20 per domain: people, nature, urban, animals, plus 20 mixed scenes) to `data_lists/e4_test_ids.txt`.

In [ ]:
run_e4("make-lists")

## 7. Stage 2: Quick Smoke Test (`--limit 2`)

Validates the entire pipeline end-to-end on GPU with 2 test images before launching the full run.

In [ ]:
run_e4("run", "--limit", "2", "--save-images")

## 8. Stage 3: Full E4 Experiment Execution

Runs all E4 conditions (default pipeline, context margins 0.0, 0.1, 0.2, 0.3, blending off, smoothing widths, bounding boxes, and E3 whole-image baseline).

*Note: Results are appended row-by-row and are fully resumable; re-running resumes from where it left off.*

In [ ]:
# Run full experiment across the 100 test images
run_e4("run", "--save-images")

## 9. Stage 4: Results Summary Table

Display quantitative metric averages across all experimental conditions.

In [ ]:
import pandas as pd

csv_path = Path("results/e4/results.csv")
if csv_path.is_file():
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} total metric records.")
    
    # Pivot mean by condition and metric
    summary_df = df.groupby(["condition", "metric"])["value"].mean().unstack()
    display(summary_df.round(4))
else:
    print("Results CSV not found yet.")

## 10. Stage 5: Generate Figures and Markdown Analysis Report

In [ ]:
run_e4("figures")

## 11. Stage 6: Display Generated Charts and Qualitative Visual Grids

In [ ]:
from IPython.display import Image as IPImage, Markdown, display

fig_dir = Path("results/e4")

for plot_name in (
    "ablation_margins.png",
    "ablation_smoothing.png",
    "ablation_components.png",
    "comparison_e3_e4.png",
    "e4_examples_grid.png",
):
    p = fig_dir / plot_name
    if p.is_file():
        print(f"=== {plot_name} ===")
        display(IPImage(filename=str(p)))

notes_file = fig_dir / "e4_notes.md"
if notes_file.is_file():
    display(Markdown(notes_file.read_text(encoding="utf-8")))

## 12. Stage 7: Download the Complete Experiment Results Bundle

Packages `results/e4` (CSV records, plots, tables, report, and sample images) into `/kaggle/working/e4_results.zip` for easy one-click download.

In [ ]:
staging = Path("/kaggle/working/e4_bundle")
shutil.rmtree(staging, ignore_errors=True)
staging.mkdir(parents=True, exist_ok=True)

if Path("results/e4").is_dir():
    shutil.copytree("results/e4", staging / "results/e4")
if Path("data_lists").is_dir():
    shutil.copytree("data_lists", staging / "data_lists")

archive = shutil.make_archive("/kaggle/working/e4_results", "zip", root_dir=staging)
print(f"Results archive ready: {archive} ({Path(archive).stat().st_size / 1e6:.2f} MB)")